# Chapter 3 / Paper 2

## Notebook 1: Mobility Preprocessing

This notebook documents the preparation and aggregation of consumer mobility measures at the census-tract level for the August 2024 analytical window.

> **Public analytical copy.** Restricted mobility records are not included. Run the notebook from the `chapter_3_paper_2` directory after placing authorized inputs in the project-relative locations described in `data/README.md`.

In [ ]:
from pathlib import Path

CHAPTER_DIR = Path.cwd().resolve()
if CHAPTER_DIR.name == 'notebooks':
    CHAPTER_DIR = CHAPTER_DIR.parent
if not (CHAPTER_DIR / 'data').exists():
    raise RuntimeError(
        'Start Jupyter from the chapter_3_paper_2 directory or its notebooks directory.'
    )

# Original analytical code below uses paths relative to the chapter directory.
import os
os.chdir(CHAPTER_DIR)


In [ ]:
## Bloco 1: Setup & Imports

In [ ]:
# Title: Setup and Configuration
import pandas as pd
import numpy as np
import os
import gzip
import sys
import multiprocessing as mp
from tqdm import tqdm

# Ensure real-time output in Jupyter
from IPython.display import display, HTML
display(HTML("<style>.jp-OutputArea{font-size: 14px;}</style>"))

# Path to raw mobility data
MOBILITY_PATH = "data/private/mobility/"
MOBILITY_FILE = os.path.join(MOBILITY_PATH, "mobilidade_unificada_Aug_2024.csv.gz")

# Output folder for intermediate files
OUTPUT_PATH = os.path.join(MOBILITY_PATH, "processed_aug2024")
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Environment set up. Ready to load mobility data.")

In [ ]:
## Block 2: Reading and pre visualization

In [ ]:
# Title: Load Mobility Data (August 2024)

chunks = []
with gzip.open(MOBILITY_FILE, 'rt') as f:
    first_chunk = pd.read_csv(f, nrows=100_000)
    print("Preview of mobility data (first 5 rows):")
    display(first_chunk.head())
    print(f"\nColumns: {list(first_chunk.columns)}")

In [ ]:
## Bloco 3: Clean and Save Raw Mobility

In [ ]:
# Title: High-Efficiency Chunked Cleaning and Direct Save

import pandas as pd
import os

MOBILITY_FILE = "data/private/mobility/mobilidade_unificada_Aug_2024.csv.gz"
OUTPUT_PATH = "data/private/mobility/processed_aug2024"
os.makedirs(OUTPUT_PATH, exist_ok=True)
clean_output_path = os.path.join(OUTPUT_PATH, "mobility_aug2024_clean.csv.gz")

chunk_size = 2_000_000
row_count = 0
is_first = True

print("⏳ Starting streaming read, clean, and save...")

for chunk in pd.read_csv(MOBILITY_FILE, compression='gzip', low_memory=False, chunksize=chunk_size):
    initial_rows = len(chunk)
    chunk = chunk.dropna(subset=['store_id', 'month_part'])
    row_count += len(chunk)

    # Write to disk incrementally
    chunk.to_csv(clean_output_path, mode='w' if is_first else 'a',
                 header=is_first, index=False, compression='gzip')
    is_first = False

    print(f" Processed chunk: {initial_rows} → {len(chunk)} rows written")

print(f"\n Total rows after cleaning: {row_count:,}")
print(f" Final cleaned file saved at: {clean_output_path}")

In [ ]:
## Block 4 — Aggregation by setor with high performance

In [ ]:
# Title: Ultra-Fast Aggregation by Sector (Vectorized Version)

import pandas as pd
import os

clean_input_path = "data/private/mobility/processed_aug2024/mobility_aug2024_clean.csv.gz"
agg_output_path = "data/private/mobility/aggregated_aug2024"
os.makedirs(agg_output_path, exist_ok=True)
agg_file = os.path.join(agg_output_path, "mobility_by_sector.csv.gz")

chunk_size = 2_000_000
agg_chunks = []

print("⏳ Starting vectorized aggregation by store_id...")

for chunk in pd.read_csv(clean_input_path, chunksize=chunk_size, low_memory=False):
    chunk_grouped = chunk.groupby('store_id').agg({
        'unique': 'sum',
        'visits': 'sum',
        'raw_unique': 'sum',
        'raw_visits': 'sum',
        'repeat_visitors': 'sum',
        'new_visitors': 'sum',
        'dwell_time_mins': ['sum', 'count'],
        'state': 'first'
    })
    agg_chunks.append(chunk_grouped)

# Combine all intermediate results
combined = pd.concat(agg_chunks).groupby('store_id').agg({
    ('unique', 'sum'): 'sum',
    ('visits', 'sum'): 'sum',
    ('raw_unique', 'sum'): 'sum',
    ('raw_visits', 'sum'): 'sum',
    ('repeat_visitors', 'sum'): 'sum',
    ('new_visitors', 'sum'): 'sum',
    ('dwell_time_mins', 'sum'): 'sum',
    ('dwell_time_mins', 'count'): 'sum',
    ('state', 'first'): 'first'
})

# Flatten columns
combined.columns = [
    'total_unique_visitors', 'total_visits', 'raw_unique_visitors', 'raw_total_visits',
    'total_repeat_visitors', 'total_new_visitors', 'dwell_sum', 'dwell_count', 'state'
]
combined.reset_index(inplace=True)
combined['avg_dwell_time_mins'] = combined['dwell_sum'] / combined['dwell_count']

# Final format
final_df = combined[[
    'store_id', 'total_unique_visitors', 'total_visits', 'raw_unique_visitors',
    'raw_total_visits', 'total_repeat_visitors', 'total_new_visitors',
    'avg_dwell_time_mins', 'state'
]].rename(columns={'store_id': 'code_censo'})

# Save final file
final_df.to_csv(agg_file, index=False, compression='gzip')
print(f" Aggregated file saved at: {agg_file}")

In [ ]:
##  Block 5: Full Aggregation of All Numeric Columns by Sector

In [ ]:
# Title: Full Aggregation of All Numeric Columns by Sector (No Re-Groupby)

import pandas as pd
import numpy as np
import os

# Paths
clean_input_path = "data/private/mobility/processed_aug2024/mobility_aug2024_clean.csv.gz"
agg_output_path = "data/private/mobility/aggregated_aug2024"
os.makedirs(agg_output_path, exist_ok=True)
agg_file = os.path.join(agg_output_path, "mobility_by_sector_full.csv.gz")

# Config
chunk_size = 2_000_000
aggregated_chunks = {}

print("⏳ Starting robust full numeric aggregation by sector...")

for chunk in pd.read_csv(clean_input_path, chunksize=chunk_size, low_memory=False):
    numeric_cols = chunk.select_dtypes(include=[np.number]).columns.tolist()
    grouped = chunk.groupby("store_id")[numeric_cols].agg(['sum', 'mean'])

    # Aggregate into dict
    for store_id, row in grouped.iterrows():
        if store_id not in aggregated_chunks:
            aggregated_chunks[store_id] = row
        else:
            aggregated_chunks[store_id] += row

# Convert back to DataFrame
final = pd.DataFrame.from_dict(aggregated_chunks, orient='index')
final.index.name = 'code_censo'
final.reset_index(inplace=True)

# Flatten MultiIndex columns
final.columns = ['code_censo'] + [f"{col}_{stat}" for col, stat in final.columns[1:]]

# Save to disk
final.to_csv(agg_file, index=False, compression='gzip')
print(f" Full numeric aggregation saved at: {agg_file}")

In [ ]:
## Bloco 6: Sociodemographic Aggregation by Sector

In [ ]:
# Title: Sociodemographic Aggregation by Sector (Proportions)

import pandas as pd
import os
from collections import defaultdict

# Paths
clean_input_path = "data/private/mobility/processed_aug2024/mobility_aug2024_clean.csv.gz"
agg_output_path = "data/private/mobility/aggregated_aug2024"
os.makedirs(agg_output_path, exist_ok=True)
output_file = os.path.join(agg_output_path, "mobility_sociodemographics.csv.gz")

# Config
chunk_size = 2_000_000
agg_data = defaultdict(lambda: defaultdict(int))

print("⏳ Starting sociodemographic aggregation...")

for chunk in pd.read_csv(clean_input_path, chunksize=chunk_size, low_memory=False):
    for var in ['demographics_gender', 'demographics_age_range', 'demographics_class']:
        grouped = chunk.groupby('store_id')[var].value_counts()
        for (store_id, category), count in grouped.items():
            col_name = f"{var}_{category}"
            agg_data[store_id][col_name] += count

print(" Aggregation done. Converting to DataFrame...")

# Transform to DataFrame
records = []
for store_id, counts in agg_data.items():
    total = sum(counts.values())
    row = {'code_censo': store_id}
    for col, val in counts.items():
        row[col] = val / total  # proportion
    records.append(row)

final_df = pd.DataFrame(records)
final_df.to_csv(output_file, index=False, compression='gzip')
print(f" Sociodemographic proportions saved at: {output_file}")

In [ ]:
## Bloco 6: Sociodemographic Aggregation by Sector